# Experiment 3
Creation of the ordinal and sequential refinements

In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
import json

import preprocessing as pp
import candidates as ca
import refinements as rf
import subgroup_selection as ss
import search as bs

## Data cleaning and filtering

In [2]:
unclean_data = pd.read_csv('algebrakit_firstbatch_2.csv')
df = pp.clean_data_algebrakit(unfilterd_data=unclean_data, filter_type="remove_interaction")
print(f"we have {len(df)} records of {len(df['sessionID'].unique())} sessions with {df['finished'].value_counts()}")

we have 6020 records of 1622 sessions with finished
False    3958
True     2062
Name: count, dtype: int64


## Beam search with Collection and Ordinal Refinements implemented

In [4]:
descriptors = [ 'itemType', 'interactionType', #categorical
               'skills', 'positive_skills', 'negative_skills', #collection 
               'event_types', 'event_evaluations',  #sequence
               'stepTypes', #subsequence
               'marksTotal', 'marksTotal_interaction', 'usedAttempts', 'marksPenalty', 'hintsRequested', 'mathErrors'] #ordinal


targets = ['finished'] # all to do with performance

ids = ['exerciseID', 'sessionID', 'questionID', 'interactionID', 'representative_index']

# +- 20 seconden (with naive collection this is 12 mins)
refinement_params = {
                     'numerical_bins': 3,
                     'minimal_support_collections': 0.2, 'maximum_support_collections': 0.9, 'maximum_length_collections': 3,
                     'minimal_support_sequence': 0.2, 'maximum_support_sequence': 0.9, 'maximum_length_sequence': 3,
                     'minimal_support_subsequence': 0.2, 'maximum_support_subsequence': 0.9, 'maximum_length_subsequence': 3,
                     }

beam_params = {'k': 10, 'd': 3, 'w': 20}
constraint_params = {'minimum_size': 25}
model_params = {'model': 'F_beta', 'beta': 1, 'weighted': True, 'entropy level': 'exercise'}

result_set = bs.beam_search(data=df, descriptors=descriptors, targets=targets, ids=ids, 
                            refinement_params=refinement_params, beam_params=beam_params,
                            constraint_params=constraint_params, model_params=model_params)

print(len(result_set._queue))

Search started at 2025-12-02 15:43:59.174633
start on level 1 after 0:00:01.278696
start on level 2 after 0:00:11.081774
start on level 3 after 0:00:16.001329
This search took 0:00:18.477248
10


In [5]:
for sub in result_set._queue:
    print(sub)
    # print(f" avg. recall = {np.mean(sub._metrics['recall'])}")
    # print(f" avg. precision = {np.mean(sub._metrics['precision'])}")
    print(f" avg. F_beta = {np.mean(sub._metrics['F_beta'])}")
    print(f" avg. goodness_weights = {np.mean(sub._metrics['goodness_weights'])}")
    print(f" avg. fail rate = {np.mean(sub._metrics['fail rates'])}")
    # print(f" entropy = {sub._metrics['entropy']}")
    print(f" sessions = {sub._metrics['session_counts']}")
    print(56 * "=========")

Records with INTERACTIONTYPE is FILL_IN_THE_BLANKS & MARKSTOTAL is between 4.0 and 6.0 have a quality score of 0.11051408879808258.
 avg. F_beta = 0.9060997732426304
 avg. goodness_weights = 0.8354838709677419
 avg. fail rate = 0.8472222222222222
 sessions = [108, 117]
Records with INTERACTIONTYPE is FILL_IN_THE_BLANKS & MARKSTOTAL is between 4.0 and 4.0 have a quality score of 0.09636939203400273.
 avg. F_beta = 0.9244444444444444
 avg. goodness_weights = 0.8709677419354839
 avg. fail rate = 0.8888888888888888
 sessions = [117]
Records with SKILLS contains at least ['EquationsOperationsAdd', 'AlgebraAddingLikeTerms'] & EVENT_TYPES contains at least ['EVALUATE', 'GIVEUP'] & MARKSTOTAL is between 4.0 and 6.0 & EVENT_TYPES contains at least ['EVALUATE', 'GIVEUP'] have a quality score of 0.09587332401202643.
 avg. F_beta = 0.9686625985252988
 avg. goodness_weights = 0.6797619047619048
 avg. fail rate = 0.9939759036144578
 sessions = [83, 45]
Records with MARKSTOTAL is between 4.0 and 8.0 